# EDA — Observatório de Saúde do Trabalhador (Recife)

Conecta no PostgreSQL do Aiven usando as credenciais do `.env` na raiz do projeto (mesmas variáveis do `pipeline/03_carga.py`)
e responde às perguntas analíticas do canvas (`docs/canvas-projeto.md`, seção 3).

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, inspect, text

# usecwd=True: o Jupyter roda a partir de analises/, então sobe até achar o .env da raiz.
load_dotenv(find_dotenv(usecwd=True))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?sslmode=require",
    pool_pre_ping=True,  # reabre a conexão se o Aiven derrubar por inatividade
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)


def consulta(sql, **params):
    """Roda um SELECT e devolve um DataFrame."""
    with engine.connect() as conexao:
        return pd.read_sql(text(sql), conexao, params=params)

## Visão geral das tabelas

In [2]:
tabelas = inspect(engine).get_table_names()
contagens = consulta(
    " UNION ALL ".join(f"SELECT '{t}' AS tabela, COUNT(*) AS linhas FROM {t}" for t in tabelas)
)
contagens.sort_values("tabela")

,tabela,linhas
0,dim_cbo,2726
4,dim_cid10,14941
1,dim_cnae,674
5,fato_caged,1272132
7,fato_cat,14020
2,fato_rais,533143
3,fato_sinan_acbi,5424
6,fato_sinan_acgr,3650


## Cuidados antes de interpretar

1. **Cobertura irregular da CAT.** Os arquivos mensais do INSS não têm volume regular: set/2024–mai/2025 e nov–dez/2025
   chegam com uma fração do volume normal, e antes de jun/2023 só há CATs registradas com atraso. A consulta Q0 mostra
   a cobertura mês a mês. Comparações entre anos usam só os meses com cobertura adequada.
2. **Território:** as fontes de acidente só têm o município (Recife). Não há bairro.
3. **CNAE:** CAT e RAIS são cruzadas no nível de 4 dígitos (`dim_cnae.classe_codigo_4`).
4. **"Adoecimentos":** os agravos de doença do SINAN não estão carregados; o tipo "Doença" da CAT é a aproximação.
5. **Denominador:** usamos vínculos **CLT** da RAIS. Servidores estatutários têm regime próprio e não geram CAT.
6. **Período:** 2023–2025, os três anos em que CAT e RAIS coexistem. A CAT vai até 2026, mas a RAIS não.
7. **`{ñ class}`** é o marcador do INSS para "não classificado" em alguns campos de texto da CAT. Trate como ausente.

A regra de cobertura se repete em várias consultas, então fica numa função:

In [3]:
def cobertura(inicio="2023-01-01", fim="2025-12-01"):
    """CTEs meses/mensal/cobertura: mês adequado = pelo menos metade da mediana mensal do período."""
    return f"""
meses AS (
  SELECT generate_series('{inicio}'::date, '{fim}'::date, interval '1 month')::date AS mes
),
mensal AS (
  SELECT m.mes, count(f.id) AS n
  FROM meses m
  LEFT JOIN fato_cat f ON date_trunc('month', f.data_acidente)::date = m.mes
  GROUP BY m.mes
),
cobertura AS (
  SELECT mes, n,
         n >= 0.5 * (SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY n) FROM mensal) AS ok
  FROM mensal
)"""

### [Q0] Cobertura mensal da CAT

In [4]:
q0 = consulta(f"""
WITH {cobertura()}
SELECT to_char(mes, 'YYYY-MM') AS mes, n AS acidentes_cat, ok AS cobertura_adequada
FROM cobertura
ORDER BY mes
""")
q0

,mes,acidentes_cat,cobertura_adequada
0,2023-01,8,False
1,2023-02,7,False
2,2023-03,13,False
3,2023-04,26,False
4,2023-05,77,False
5,2023-06,418,True
6,2023-07,439,True
7,2023-08,482,True
8,2023-09,408,True
9,2023-10,436,True


---
## Pergunta 1
**Quais atividades econômicas (CNAE) concentram mais acidentes e adoecimentos do trabalho no Recife, proporcionalmente
ao número de trabalhadores formais, e como isso mudou nos últimos três anos?**

Taxa por mil vínculos CLT, por ano (2023–2025). Cada ano usa só os seus meses com cobertura adequada, e a taxa é
anualizada: ocorrências nesses meses ÷ (vínculos × meses adequados / 12). Supõe sazonalidade pequena, então variações
pequenas são ruído. Só entram setores com ≥ 2.000 vínculos CLT em cada ano e ≥ 30 ocorrências no total.

### [Q1a] Ranking de setores pela taxa em 2025, com a evolução desde 2023

In [5]:
q1a = consulta(f"""
WITH {cobertura()},
meses_ok AS (
  SELECT extract(year FROM mes)::int AS ano, count(*) FILTER (WHERE ok) AS n_meses
  FROM cobertura
  GROUP BY 1
),
acidentes AS (
  SELECT extract(year FROM f.data_acidente)::int AS ano,
         f.cnae AS classe4,
         count(*) AS ocorrencias,
         count(*) FILTER (WHERE btrim(f.tipo_acidente) ILIKE 'Doen%') AS doencas
  FROM fato_cat f
  JOIN cobertura c ON c.mes = date_trunc('month', f.data_acidente)::date AND c.ok
  GROUP BY 1, 2
),
vinculos AS (
  SELECT r.ano, d.classe_codigo_4 AS classe4, sum(r.quantidade_vinculos_clt) AS vinculos
  FROM fato_rais r
  JOIN dim_cnae d ON d.classe_codigo = r.cnae_2_classe
  WHERE r.ano BETWEEN 2023 AND 2025
  GROUP BY 1, 2
),
taxas AS (
  SELECT v.ano, v.classe4, v.vinculos,
         coalesce(a.ocorrencias, 0) AS ocorrencias,
         coalesce(a.doencas, 0) AS doencas,
         1000.0 * coalesce(a.ocorrencias, 0) / (v.vinculos * m.n_meses / 12.0) AS taxa
  FROM vinculos v
  JOIN meses_ok m ON m.ano = v.ano
  LEFT JOIN acidentes a ON a.ano = v.ano AND a.classe4 = v.classe4
  WHERE v.vinculos > 0
)
SELECT d.classe_codigo_4 AS cnae,
       d.denominacao AS atividade_economica,
       max(t.vinculos) FILTER (WHERE t.ano = 2025) AS vinculos_clt_2025,
       sum(t.ocorrencias) AS ocorrencias_meses_adequados,
       sum(t.doencas) AS doencas_do_trabalho,
       round(max(t.taxa) FILTER (WHERE t.ano = 2023), 1) AS taxa_2023,
       round(max(t.taxa) FILTER (WHERE t.ano = 2024), 1) AS taxa_2024,
       round(max(t.taxa) FILTER (WHERE t.ano = 2025), 1) AS taxa_2025,
       round(max(t.taxa) FILTER (WHERE t.ano = 2025)
             - max(t.taxa) FILTER (WHERE t.ano = 2023), 1) AS variacao_2023_2025
FROM taxas t
JOIN dim_cnae d ON d.classe_codigo_4 = t.classe4
GROUP BY d.classe_codigo_4, d.denominacao
HAVING count(*) = 3 AND min(t.vinculos) >= 2000 AND sum(t.ocorrencias) >= 30
ORDER BY taxa_2025 DESC
LIMIT 20
""")
q1a

,cnae,atividade_economica,vinculos_clt_2025,ocorrencias_meses_adequados,doencas_do_trabalho,taxa_2023,taxa_2024,taxa_2025,variacao_2023_2025
0,3811,Coleta de resíduos não-perigosos,3402,273.0,1.0,44.2,48.1,35.8,-8.4
1,8610,Atividades de atendimento hospitalar,51395,2250.0,16.0,24.5,22.1,26.4,1.9
2,4110,Incorporação de empreendimentos imobiliários,8878,197.0,4.0,6.8,9.6,21.8,15.1
3,4634,"Comércio atacadista de carnes, produtos da carne e pescado",4030,101.0,2.0,12.9,11.3,20.0,7.1
4,7739,Aluguel de máquinas e equipamentos não especificados ant...,2612,82.0,0.0,17.5,16.4,19.7,2.2
5,3511,Geração de energia elétrica,2771,66.0,7.0,6.9,13.2,15.5,8.6
6,8640,Atividades de serviços de complementação diagnóstica e t...,12316,246.0,1.0,8.1,9.0,14.5,6.4
7,3600,"Captação, tratamento e distribuição de água",2786,70.0,1.0,16.3,9.7,14.2,-2.2
8,4635,Comércio atacadista de bebidas,3392,77.0,14.0,14.1,10.9,13.6,-0.5
9,5320,Atividades de malote e de entrega,2202,50.0,1.0,15.1,8.5,12.5,-2.6


### [Q1b] Referência: taxa da cidade inteira por ano, pela mesma regra

In [14]:
q1b = consulta(f"""
WITH {cobertura()},
meses_ok AS (
  SELECT extract(year FROM mes)::int AS ano, count(*) FILTER (WHERE ok) AS n_meses
  FROM cobertura
  GROUP BY 1
),
acidentes AS (
  SELECT extract(year FROM f.data_acidente)::int AS ano, count(*) AS ocorrencias
  FROM fato_cat f
  JOIN cobertura c ON c.mes = date_trunc('month', f.data_acidente)::date AND c.ok
  GROUP BY 1
),
vinculos AS (
  SELECT ano, sum(quantidade_vinculos_clt) AS vinculos
  FROM fato_rais
  WHERE ano BETWEEN 2023 AND 2025
  GROUP BY 1
)
SELECT v.ano,
       m.n_meses AS meses_com_cobertura_adequada,
       v.vinculos AS vinculos_clt,
       a.ocorrencias,
       round(1000.0 * a.ocorrencias / (v.vinculos * m.n_meses / 12.0), 1) AS taxa_por_mil_vinculos_ano
FROM vinculos v
JOIN meses_ok m ON m.ano = v.ano
JOIN acidentes a ON a.ano = v.ano
ORDER BY v.ano
""")
q1b

,ano,meses_com_cobertura_adequada,vinculos_clt,ocorrencias,taxa_por_mil_vinculos_ano
0,2023,7,980240,2819,4.9
1,2024,8,1003128,3404,5.1
2,2025,7,1045441,3783,6.2


---
## Pergunta 2
**Quais ocupações (CBO) e diagnósticos (CID-10) respondem pela maior parte das CATs registradas no Recife, e essa
composição mudou entre o primeiro e o último dos três anos?**

- **"Maior parte":** ranking de todas as CATs de 2023–2025, com a participação acumulada (`% acumulado`). As linhas
  até ~50% ou ~80% são o grupo que "responde pela maior parte".
- **"Mudou entre o primeiro e o último ano":** participação em 2023 × 2025, só nos meses do calendário com cobertura
  adequada **nos dois anos**, para não confundir mudança de perfil com falha de cobertura ou sazonalidade.
  A variação é em pontos percentuais, sem teste estatístico: com poucos casos, oscilações pequenas são ruído.
- CID-10 na categoria de 3 caracteres. `upper()` porque algumas CATs vêm com o CID em minúsculas (corrigido no
  pipeline, mas o banco só muda na próxima carga).

### [Q2] Ocupações (CBO) e diagnósticos (CID-10) que concentram as CATs, e a composição em 2023 × 2025

In [15]:
q2 = consulta(f"""
WITH {cobertura()},
comparaveis AS (
  SELECT extract(month FROM mes)::int AS mes_do_ano
  FROM cobertura
  WHERE extract(year FROM mes) IN (2023, 2025)
  GROUP BY 1
  HAVING bool_and(ok) AND count(*) = 2
),
cats AS (
  SELECT EXTRACT(YEAR FROM data_acidente)::int AS ano,
         EXTRACT(YEAR FROM data_acidente) IN (2023, 2025)
           AND EXTRACT(MONTH FROM data_acidente)::int IN (SELECT mes_do_ano FROM comparaveis) AS comparavel,
         cbo_2002,
         UPPER(LEFT(REPLACE(BTRIM(cid10), '.', ''), 3)) AS cid_categoria
  FROM fato_cat
  WHERE data_acidente BETWEEN '2023-01-01' AND '2025-12-31'
),
itens AS (
  SELECT 'CBO' AS dimensao, COALESCE(cbo_2002, 'NI') AS codigo, ano, comparavel FROM cats
  UNION ALL
  SELECT 'CID-10', COALESCE(cid_categoria, 'NI'), ano, comparavel FROM cats
),
total AS (
  SELECT dimensao, codigo, COUNT(*) AS n,
         100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY dimensao) AS pct_total,
         100.0 * SUM(COUNT(*)) OVER (PARTITION BY dimensao ORDER BY COUNT(*) DESC, codigo)
               / SUM(COUNT(*)) OVER (PARTITION BY dimensao) AS pct_acumulado,
         ROW_NUMBER() OVER (PARTITION BY dimensao ORDER BY COUNT(*) DESC, codigo) AS ranking
  FROM itens
  GROUP BY dimensao, codigo
),
comp AS (
  SELECT dimensao, codigo, ano,
         100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY dimensao, ano) AS pct
  FROM itens
  WHERE comparavel
  GROUP BY dimensao, codigo, ano
),
pct AS (
  SELECT dimensao, codigo,
         MAX(pct) FILTER (WHERE ano = 2023) AS pct_2023,
         MAX(pct) FILTER (WHERE ano = 2025) AS pct_2025
  FROM comp
  GROUP BY dimensao, codigo
),
descr AS (
  SELECT 'CBO' AS dimensao, cbo_2002 AS codigo, MAX(cbo_2002_descricao) AS descricao
  FROM dim_cbo
  GROUP BY cbo_2002
  UNION ALL
  SELECT 'CID-10', categoria, MAX(descricao_categoria)
  FROM dim_cid10
  GROUP BY categoria
)
SELECT CASE t.dimensao WHEN 'CBO' THEN 'Ocupação (CBO)' ELSE 'Diagnóstico (CID-10)' END AS "Tipo de análise",
       t.ranking AS "Posição",
       t.codigo AS "Código",
       CASE WHEN t.codigo = 'NI' THEN 'Não informado'
            ELSE COALESCE(d.descricao, 'Sem descrição na tabela de referência') END AS "Descrição",
       t.n AS "CATs 2023–2025",
       ROUND(t.pct_total, 1) AS "% do total",
       ROUND(t.pct_acumulado, 1) AS "% acumulado",
       ROUND(COALESCE(p.pct_2023, 0), 1) AS "% em 2023",
       ROUND(COALESCE(p.pct_2025, 0), 1) AS "% em 2025",
       ROUND(COALESCE(p.pct_2025, 0) - COALESCE(p.pct_2023, 0), 1) AS "Variação (p.p.)"
FROM total t
LEFT JOIN pct p ON p.dimensao = t.dimensao AND p.codigo = t.codigo
LEFT JOIN descr d ON d.dimensao = t.dimensao AND d.codigo = t.codigo
WHERE t.ranking <= 20
ORDER BY t.dimensao, t.ranking
""")
q2

,Tipo de análise,Posição,Código,Descrição,CATs 2023–2025,% do total,% acumulado,% em 2023,% em 2025,Variação (p.p.)
0,Ocupação (CBO),1,322205,Técnico de enfermagem,1266,11.9,11.9,13.5,11.4,-2.1
1,Ocupação (CBO),2,514320,Faxineiro,522,4.9,16.7,5.9,4.5,-1.4
2,Ocupação (CBO),3,717020,Servente de obras,447,4.2,20.9,4.7,4.8,0.2
3,Ocupação (CBO),4,517330,Vigilante patrimonial,307,2.9,23.8,2.6,2.6,0.0
4,Ocupação (CBO),5,411010,Assistente administrativo,274,2.6,26.4,2.4,2.8,0.4
5,Ocupação (CBO),6,223505,Enfermeiro,223,2.1,28.5,1.8,2.2,0.3
6,Ocupação (CBO),7,513505,Auxiliar nos serviços de alimentação,221,2.1,30.5,2.2,2.1,-0.1
7,Ocupação (CBO),8,514205,Coletor de lixo domiciliar,215,2.0,32.5,1.7,1.9,0.2
8,Ocupação (CBO),9,521110,Vendedor de comércio varejista,188,1.8,34.3,1.5,1.7,0.3
9,Ocupação (CBO),10,411005,Auxiliar de escritório,181,1.7,36.0,1.7,1.6,-0.2


---
## Pergunta 3
**Onde há muito vínculo formal e pouca notificação — sinal de subnotificação, não de segurança?**

Mesma ideia da taxa por mil vínculos por CNAE, com três ajustes:
- vínculos **CLT** (estatutários não geram CAT; com `quantidade_vinculos_ativos` a administração pública domina o ranking);
- CAT e RAIS no **mesmo período** (2023–2025; a CAT vai até 2026);
- ordenado pela **diferença esperado − observado** (esperado = vínculos do setor × taxa da cidade), e não pela taxa
  crescente, que só lista setores pequenos ou de risco baixo com 0 CAT.

A taxa absoluta é subestimada pela cobertura da CAT, mas a taxa da cidade usada como referência sofre a mesma
subestimação. Leia como **comparação entre setores** e como indicador para investigar, não como prova. Setores de risco
realmente baixo (escritórios) também aparecem com taxa baixa.

### [Q3a] Setores com mais CATs “faltando” em relação à taxa da cidade (2023–2025)

In [16]:
q3a = consulta("""
WITH acidentes AS (
  SELECT cnae AS classe4, count(*) AS acidentes
  FROM fato_cat
  WHERE data_acidente BETWEEN '2023-01-01' AND '2025-12-31'
  GROUP BY cnae
),
vinculos AS (
  SELECT d.classe_codigo_4 AS classe4, sum(r.quantidade_vinculos_clt) AS vinculos_ano
  FROM fato_rais r
  JOIN dim_cnae d ON d.classe_codigo = r.cnae_2_classe
  WHERE r.ano BETWEEN 2023 AND 2025
  GROUP BY d.classe_codigo_4
),
geral AS (
  SELECT (SELECT sum(acidentes) FROM acidentes)::numeric
         / (SELECT sum(vinculos_ano) FROM vinculos) AS taxa
)
SELECT v.classe4 AS cnae,
       d.denominacao AS atividade_economica,
       v.vinculos_ano AS vinculos_clt_ano,
       coalesce(a.acidentes, 0) AS acidentes_cat,
       round(1000.0 * coalesce(a.acidentes, 0) / v.vinculos_ano, 2) AS acidentes_por_mil_vinculos,
       round(v.vinculos_ano * g.taxa, 0) AS acidentes_esperados,
       round(v.vinculos_ano * g.taxa - coalesce(a.acidentes, 0), 0) AS diferenca_esperado_menos_observado,
       round(coalesce(a.acidentes, 0) / (v.vinculos_ano * g.taxa), 2) AS razao_observado_esperado
FROM vinculos v
CROSS JOIN geral g
LEFT JOIN acidentes a ON a.classe4 = v.classe4
JOIN dim_cnae d ON d.classe_codigo_4 = v.classe4
WHERE v.vinculos_ano >= 3000  -- ~1.000 vínculos por ano, como na versão original
ORDER BY diferenca_esperado_menos_observado DESC
LIMIT 20
""")
q3a

,cnae,atividade_economica,vinculos_clt_ano,acidentes_cat,acidentes_por_mil_vinculos,acidentes_esperados,diferenca_esperado_menos_observado,razao_observado_esperado
0,8411,Administração pública em geral,99783,36,0.36,352.0,316.0,0.10
1,8112,Condomínios prediais,133586,231,1.73,471.0,240.0,0.49
2,8513,Ensino fundamental,46852,60,1.28,165.0,105.0,0.36
3,8211,Serviços combinados de escritório e apoio administrativo,70805,159,2.25,250.0,91.0,0.64
4,4771,Comércio varejista de produtos farmacêuticos para uso hu...,35742,41,1.15,126.0,85.0,0.33
5,8121,Limpeza em prédios e em domicílios,33666,38,1.13,119.0,81.0,0.32
6,4781,Comércio varejista de artigos do vestuário e acessórios,40962,64,1.56,144.0,80.0,0.44
7,8532,Educação superior - graduação e pós-graduação,29702,27,0.91,105.0,78.0,0.26
8,4120,Construção de edifícios,110432,311,2.82,389.0,78.0,0.80
9,9430,Atividades de associações de defesa de direitos sociais,30486,34,1.12,107.0,73.0,0.32


### [Q3b] Verificação: CATs em setores sem nenhum vínculo CLT na RAIS 2023–2025
Se esse percentual for alto, as taxas da Q3a estão distorcidas por diferença de classificação entre as bases.

In [17]:
q3b = consulta("""
WITH vinculos AS (
  SELECT DISTINCT d.classe_codigo_4 AS classe4
  FROM fato_rais r
  JOIN dim_cnae d ON d.classe_codigo = r.cnae_2_classe
  WHERE r.ano BETWEEN 2023 AND 2025 AND r.quantidade_vinculos_clt > 0
)
SELECT count(*) AS acidentes_cat,
       count(*) FILTER (WHERE v.classe4 IS NULL) AS em_setor_sem_vinculo_na_rais,
       round(100.0 * count(*) FILTER (WHERE v.classe4 IS NULL) / count(*), 2) AS pct
FROM fato_cat f
LEFT JOIN vinculos v ON v.classe4 = f.cnae
WHERE f.data_acidente BETWEEN '2023-01-01' AND '2025-12-31'
""")
q3b

,acidentes_cat,em_setor_sem_vinculo_na_rais,pct
0,10677,0,0.0


---
## Pergunta 4
**A procura pela rede de saúde acompanha o mapa do trabalho formal ou aponta para o informal?**

A procura pela rede de saúde ainda é majoritariamente registrada como de trabalhadores formais (60,66%), refletindo o mapa do trabalho formal. No entanto, os dados apontam que a informalidade já responde por mais de 1/4 (25,89%) de todos os acidentes graves atendidos no SUS do Recife, considerando os 3 ultimos anos.
Além disso, a análise revela um forte indício de subnotificação epidemiológica, impulsionada pelo alto índice de dados omissos na rede pública, em torno de 12%.


Alerta para:
1 Capacitação nas UPAs e Policlínicas: Combater o índice de 11,81% de dados não informados, acabar com a invisibilidade.


2 Políticas de Território: Criar ações de prevenção focadas nos 25,89% de informais diretamente nos grandes polos de comércio e serviços autônomos de Recife.


In [ ]:
SELECT 
    sinan.nu_ano AS ano_acidente,
    CASE 
        WHEN sinan.sit_trab = 1 THEN 'Trabalho Formal (Com Carteira Assinada)'
        WHEN sinan.sit_trab IN (2, 3, 4) THEN 'Trabalho Informal (Autônomo, Sem Carteira, Doméstico)'
        WHEN sinan.sit_trab IN (5, 6, 7) THEN 'Outros Vínculos (Militar, Servidor, Não Remunerado, Avulso)'
        ELSE 'Não Informado / Sem Registro'
    END AS categoria_trabalho,
    COUNT(sinan.id) AS total_atendimentos_sus,
    
    -- Calcula o percentual de forma justa, baseado apenas no total daquele ano específico
    ROUND(
        (COUNT(sinan.id) * 100.0) / SUM(COUNT(sinan.id)) OVER(PARTITION BY sinan.nu_ano), 
        2
    ) AS percentual_proporcao_no_ano

FROM fato_sinan_acgr sinan
WHERE sinan.mun_acid = '261160' -- Filtro do território de Recife nos dados baixados
  AND sinan.nu_ano IN (2023, 2024, 2025)
GROUP BY 
    sinan.nu_ano,
    CASE 
        WHEN sinan.sit_trab = 1 THEN 'Trabalho Formal (Com Carteira Assinada)'
        WHEN sinan.sit_trab IN (2, 3, 4) THEN 'Trabalho Informal (Autônomo, Sem Carteira, Doméstico)'
        WHEN sinan.sit_trab IN (5, 6, 7) THEN 'Outros Vínculos (Militar, Servidor, Não Remunerado, Avulso)'
        ELSE 'Não Informado / Sem Registro'
    END
ORDER BY 
    ano_acidente ASC, 
    total_atendimentos_sus DESC;

### [Q4a] Mapeamento dos top 20 Cids que aparecem na rede de saúde.

O Top 20 de CIDs prova que a informalidade e a formalidade em Recife machucam de formas totalmente distintas. O Trabalho Informal está majoritariamente exposto à violência urbana e viária das duas rodas, que esmaga membros inferiores (CID S82) nas avenidas da cidade. Já o Trabalho Formal, protegido pelas barreiras e fiscalizações da CLT no trânsito, concentra seus acidentes graves em ambientes operacionais internos, afetando a manipulação de máquinas e ferramentas com lesões em punhos e mãos (CID S62). A procura pelo SUS, portanto, não apenas aponta para a informalidade, mas desenha com precisão a anatomia da precarização laboral no Recife.



In [ ]:
SELECT
    sinan.nu_ano AS ano_acidente,
   
    -- Classificação do Vínculo Baseada nas Regras de Negócio do SUS
    CASE
        WHEN sinan.sit_trab = 1 THEN 'Trabalho Formal'
        WHEN sinan.sit_trab IN (2, 3, 4) THEN 'Trabalho Informal'
        ELSE 'Outros / Não Informado'
    END AS categoria_vinculo,
   
    -- Código do CID-10
    COALESCE(sinan.cid_lesao, 'N/I') AS cid_codigo,
   
    -- Descrição do Diagnóstico buscando na dimensão (priorizando subcategoria, depois categoria)
    COALESCE(cid.descricao_subcategoria, cid.descricao_categoria, 'Descrição não encontrada no dicionário') AS diagnostico_lesao,
   
    -- Volumetria (Contagem Absoluta)
    COUNT(sinan.id) AS total_casos,
   
    -- Percentual de representação dentro do total geral de acidentes daquele ano específico
    ROUND(
        (COUNT(sinan.id) * 100.0) / SUM(COUNT(sinan.id)) OVER(PARTITION BY sinan.nu_ano),
        2
    ) AS percentual_no_ano


FROM fato_sinan_acgr sinan


-- Duplo JOIN na dim_cid10 para garantir o acoplamento de códigos de 3 ou 4 dígitos
LEFT JOIN dim_cid10 cid
    ON sinan.cid_lesao = cid.subcategoria
    OR sinan.cid_lesao = cid.categoria


-- Filtros Estritos baseados no Escopo de Recife e Período de 3 anos
WHERE sinan.mun_acid = '261160'              -- Código INSS/SUS do Recife (6 dígitos)
  AND sinan.nu_ano IN (2023, 2024, 2025)       -- Recorte temporal definido no Canvas


-- Agrupamento obrigatório para colunas de agregação
GROUP BY
    sinan.nu_ano,
    CASE
        WHEN sinan.sit_trab = 1 THEN 'Trabalho Formal'
        WHEN sinan.sit_trab IN (2, 3, 4) THEN 'Trabalho Informal'
        ELSE 'Outros / Não Informado'
    END,
    sinan.cid_lesao,
    cid.descricao_subcategoria,
    cid.descricao_categoria


-- Ordenação pelo maior volume de casos para extrair a liderança do ranking
ORDER BY
    total_casos DESC
LIMIT 20;
